# Chạy tiếp phân loại 500 gene / nhóm filter (phần bị cắt do giới hạn 12h của Kaggle)

Lần chạy trước của `clf_gridsearch_fs_group.ipynb` với `FS_DIR = .../500/filter` dừng ở mốc 12h,
`classification_500_filter.zip` mới có **712 / 825 dòng** (15 phương pháp FS × 5 fold × 11 mô hình):

| fs_run | đã có | còn thiếu |
|---|---|---|
| 12 fs_run của gse68465, tcga_luad, `fs_f_classif_tcga_lusc`, `fs_mrmr_tcga_lusc` | 55/55 | — |
| `fs_multisurf_tcga_lusc` | 52/55 | fold 5: `xgboost`, `lightgbm`, `stacking` |
| `fs_mutual_info_tcga_lusc` | 0/55 | toàn bộ |
| `fs_relieff_tcga_lusc` | 0/55 | toàn bộ |

Ước lượng thời gian (theo `tuning_time_s` của lần trước, mỗi fs_run tcga_lusc ≈ 1,0h, XGBoost chiếm ~2/3):
≈ 0,15h + 2 × 1,0h ≈ **2,2h** → chạy xong trong 1 phiên.

Notebook này giữ **nguyên** mô hình, lưới tham số, cách chọn cấu hình và định dạng đầu ra của notebook gốc; chỉ thêm:

- **Đầu vào 3 — `PREV_RESULTS`**: kết quả lần trước (`results_filter.csv`, file zip, hoặc thư mục chứa nó).
  File được chép vào `RESULTS_CSV` trước khi chạy → cơ chế "bỏ qua phần đã xong" của notebook gốc tự chạy tiếp
  đúng chỗ còn thiếu. Stacking của `fs_multisurf_tcga_lusc` fold 5 dùng cấu hình base tốt nhất (MLP, RF, KNN, SVM)
  lấy từ các dòng đã có của lần trước, đúng như khi chạy liền một mạch.
- `FS_RUNS_TO_RUN` = 3 fs_run còn thiếu (chỉ giải nén/đọc những gì cần).
- **Chốt thời gian `TIME_BUDGET_H`**: hết giờ thì dừng vòng lặp nhưng vẫn ghi summary + zip,
  để lần sau chỉ cần đưa zip mới vào `PREV_RESULTS` và chạy lại.

**Đầu ra**: `/kaggle/working/classification_500_filter.zip` chứa `results_filter.csv` **đầy đủ** (dòng cũ + mới)
và `summary_filter.csv` tính trên toàn bộ 15 phương pháp FS → thay thế trực tiếp file zip cũ.

**Trên Kaggle**: Add Input 3 nguồn — kết quả FS (`fs-experiment-result`), dữ liệu H5 (`experiment-dataset`),
và dataset chứa `classification_500_filter.zip` lần trước (Kaggle tự giải nén zip khi upload).


In [ ]:
import sys, subprocess, importlib.util
# tables: backend cho pd.read_hdf | xgboost, lightgbm: Kaggle thuong co san
_missing = [p for p, m in [("tables", "tables"), ("xgboost", "xgboost"), ("lightgbm", "lightgbm")]
            if importlib.util.find_spec(m) is None]
if _missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

## 1. Cấu hình

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Dau vao 1: thu muc 1 nhom FS (chua cac zip fs_<method>_<bo>.zip hoac thu muc da giai nen)
FS_DIR = "/kaggle/input/fs-experiment-result/500/filter"
# Dau vao 2: noi chua du lieu H5 cua cac bo -> DATA_ROOT/<bo>/fold_k/{train,val,test}.h5
DATA_ROOT = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5"
OUT_ROOT = "/kaggle/working/classification"
# Dau vao 3: ket qua lan chay truoc (results_filter.csv | .zip | thu muc chua no) -- sua theo ten dataset da upload
PREV_RESULTS = "/kaggle/input/classification-500-filter"

KNOWN_DATASETS = ["gse68465", "tcga_luad", "tcga_lusc"]   # hau to ten de tach ten bo
N_FOLDS = 5
SEED = 42
N_JOBS = -1
SELECT_METRIC = "f1_macro"    # do do tren val de chon cau hinh
QUICK = False                 # True: luoi nho de chay thu nhanh
MODELS_TO_RUN = ["knn", "gaussian_nb", "logistic_regression", "decision_tree", "random_forest",
                 "linear_svm", "rbf_svm", "mlp", "xgboost", "lightgbm", "stacking"]
# 3 fs_run con thieu (None = tat ca; phan da xong van duoc bo qua nho RESULTS_CSV)
FS_RUNS_TO_RUN = ["fs_multisurf_tcga_lusc", "fs_mutual_info_tcga_lusc", "fs_relieff_tcga_lusc"]
TIME_BUDGET_H = 11.0          # het gio -> dung grid search, van ghi summary + zip (Kaggle cat o 12h)
import time
NOTEBOOK_T0 = time.time()

FS_GROUP = os.path.basename(os.path.normpath(FS_DIR))                       # vd "embedded"
N_FEATURES_TAG = os.path.basename(os.path.dirname(os.path.normpath(FS_DIR)))  # vd "100"
OUT_DIR = os.path.join(OUT_ROOT, N_FEATURES_TAG, FS_GROUP)
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUT_DIR, f"results_{FS_GROUP}.csv")
SUMMARY_CSV = os.path.join(OUT_DIR, f"summary_{FS_GROUP}.csv")

assert os.path.isdir(FS_DIR), f"Khong thay FS_DIR={FS_DIR} -- da Add Input chua?"
print("Nhom FS :", FS_GROUP, "| so gene:", N_FEATURES_TAG, "|", FS_DIR)
print("Du lieu :", DATA_ROOT)
print("Ket qua :", RESULTS_CSV)
print("So CPU  :", os.cpu_count())

## 1b. Nạp kết quả lần trước vào `RESULTS_CSV`

Gộp kết quả cũ (bỏ trùng theo `(fs_run, fold, model)`) vào `RESULTS_CSV`.
Nếu `RESULTS_CSV` đã có sẵn (chạy lại cell trong cùng phiên) thì gộp chung, không mất phần mới.


In [ ]:
import glob, zipfile, io
import pandas as pd

KEY = ["fs_run", "fold", "model"]


def load_prev_results(path):
    # path: file csv | file zip | thu muc (tim results_<nhom>.csv hoac zip ben trong)
    name = f"results_{FS_GROUP}.csv"
    if os.path.isdir(path):
        hits = (sorted(glob.glob(os.path.join(path, "**", name), recursive=True))
                or sorted(glob.glob(os.path.join(path, "**", f"classification_{N_FEATURES_TAG}_{FS_GROUP}.zip"),
                                    recursive=True)))
        assert hits, f"Khong thay {name} (hay zip) trong {path}"
        return load_prev_results(hits[0])
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as zf:
            members = [m for m in zf.namelist() if m.endswith(name)]
            assert members, f"Khong thay {name} trong {path}"
            print("Doc", path, "->", members[0])
            return pd.read_csv(io.BytesIO(zf.read(members[0])))
    print("Doc", path)
    return pd.read_csv(path)


prev_res = load_prev_results(PREV_RESULTS)
_prev_tag = (set(prev_res["n_features"].astype(str)), set(prev_res["fs_group"].astype(str)))
assert _prev_tag == ({N_FEATURES_TAG}, {FS_GROUP}), (
    f"PREV_RESULTS co (so gene, nhom) = {_prev_tag}, nhung FS_DIR cho ra ({N_FEATURES_TAG!r}, {FS_GROUP!r}). "
    f"FS_DIR phai ket thuc bang .../<so gene>/<nhom>, vd .../500/filter. FS_DIR hien tai: {FS_DIR}")
if os.path.exists(RESULTS_CSV):
    prev_res = pd.concat([prev_res, pd.read_csv(RESULTS_CSV)], ignore_index=True)
prev_res = prev_res.drop_duplicates(KEY, keep="last")
prev_res.to_csv(RESULTS_CSV, index=False)
print(f"Da nap {len(prev_res)} dong vao {RESULTS_CSV}")
print(prev_res.groupby("fs_run").size().rename("so dong").to_string())


## 2. Tìm các phương pháp FS trong `FS_DIR` (giải nén zip nếu cần)

In [ ]:
import glob, json, time, zipfile, itertools, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}
EXTRACT_DIR = os.path.join("/kaggle/working/_fs_extracted", N_FEATURES_TAG, FS_GROUP)


def parse_run_name(entry):
    # "fs_lasso_gse68465(.zip)" -> ("fs_lasso_gse68465", "lasso", "gse68465")
    stem = entry[:-4] if entry.endswith(".zip") else entry
    for ds in sorted(KNOWN_DATASETS, key=len, reverse=True):
        if stem.endswith("_" + ds):
            method = stem[:-len(ds) - 1]
            return stem, (method[3:] if method.startswith("fs_") else method), ds
    return None


def find_fold_files(root):
    # fold_k.csv nam trong selected_genes/ (ket qua FS) hoac ngay trong thu muc (selected_features/).
    # Khong lay history/ hay population_state_history/ (wrapper cung co fold_k.csv o do)
    out = {}
    for r in range(1, N_FOLDS + 1):
        hits = (sorted(glob.glob(os.path.join(root, "**", "selected_genes", f"fold_{r}.csv"), recursive=True))
                or sorted(glob.glob(os.path.join(root, "**", f"fold_{r}.csv"), recursive=True)))
        hits = [h for h in hits if "history" not in os.path.relpath(h, root)]
        assert hits, f"Khong thay fold_{r}.csv trong {root}"
        out[r] = hits[0]
    return out


FS_RUNS = {}
for entry in sorted(os.listdir(FS_DIR)):   # thu muc "x" dung truoc "x.zip" -> uu tien ban da giai nen
    path = os.path.join(FS_DIR, entry)
    parsed = parse_run_name(entry)
    if parsed is None or parsed[0] in FS_RUNS:
        continue
    stem, method, ds = parsed
    if FS_RUNS_TO_RUN is not None and stem not in FS_RUNS_TO_RUN:
        continue
    if entry.endswith(".zip"):
        root = os.path.join(EXTRACT_DIR, stem)
        if not os.path.isdir(root):
            # chi giai nen selected_genes/ (zip wrapper co population_state_history/ ~1GB/fold)
            with zipfile.ZipFile(path) as zf:
                zf.extractall(root, members=[m for m in zf.namelist()
                                             if "selected_genes/" in m or m.endswith("config.json")])
    elif os.path.isdir(path):
        root = path
    else:
        continue
    FS_RUNS[stem] = {"method": method, "dataset": ds, "folds": find_fold_files(root)}

assert FS_RUNS, f"Khong tim thay phuong phap FS nao trong {FS_DIR}"
for ds in sorted({v["dataset"] for v in FS_RUNS.values()}):
    assert os.path.isfile(os.path.join(DATA_ROOT, ds, "fold_1", "train.h5")), \
        f"Khong thay {DATA_ROOT}/{ds}/fold_1/train.h5"
if FS_RUNS_TO_RUN is not None:
    assert set(FS_RUNS_TO_RUN) <= set(FS_RUNS), f"Thieu trong FS_DIR: {set(FS_RUNS_TO_RUN) - set(FS_RUNS)}"
_n_done = prev_res.groupby("fs_run").size()
print(pd.DataFrame([{"fs_run": k, "method": v["method"], "dataset": v["dataset"],
                     "da_co": int(_n_done.get(k, 0))}
                    for k, v in FS_RUNS.items()]).to_string(index=False))


def read_genes(path):
    df = pd.read_csv(path)
    if "rank" in df.columns:
        df = df.sort_values("rank")
    return df["gene"].astype(str).tolist()

## 3. Đọc dữ liệu

In [ ]:
_DATA_CACHE = {}


def load_dataset(ds):
    # Chi giu 1 bo trong RAM; cac fs_run duoc chay theo thu tu bo de doc moi bo 1 lan
    if ds not in _DATA_CACHE:
        _DATA_CACHE.clear()
        _DATA_CACHE[ds] = {r: {s: pd.read_hdf(os.path.join(DATA_ROOT, ds, f"fold_{r}", f"{s}.h5"), key="data")
                               for s in ["train", "val", "test"]}
                           for r in range(1, N_FOLDS + 1)}
    return _DATA_CACHE[ds]


def get_xy(df, genes):
    missing = [g for g in genes if g not in df.columns]
    assert not missing, f"{len(missing)} gene khong co trong du lieu, vd {missing[:5]}"
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return df[genes].values.astype(np.float64), df["label"].values.astype(int)

## 4. Mô hình và lưới tham số

Lưới lấy từ `model_tuning_5fold_kaggle.ipynb`, chỉnh lại khoảng `C`/`gamma` cho phù hợp dữ liệu 100 gene
(lưới cũ dò cho ~20.000 gene: `gamma="scale"` giờ cỡ 1e-3 thay vì 1e-5). Cấu hình mặc định luôn được chạy kèm.
XGBoost không có `class_weight` nên `class_weight="balanced"` được thực hiện bằng `sample_weight`.

**Stacking** (theo bài báo): base learner = MLP, RF, KNN, SVM — dùng **cấu hình tốt nhất trên val của
cùng fold** (SVM = linear hoặc RBF, cái nào val macro-F1 cao hơn); meta-learner = XGBoost, học trên dự đoán
out-of-fold (StratifiedKFold trên train) của base learner; chỉ dò lưới tham số của meta-learner.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SVM_MAX_ITER = 1_000_000   # tranh libsvm chay vo han tren du lieu khong chuan hoa


def build_model(model, params):
    p = dict(params)
    if model == "knn":
        return KNeighborsClassifier(n_jobs=N_JOBS, **p)
    if model == "gaussian_nb":
        return GaussianNB(**p)
    if model == "logistic_regression":
        return LogisticRegression(max_iter=5000, random_state=SEED, **p)
    if model == "decision_tree":
        return DecisionTreeClassifier(random_state=SEED, **p)
    if model == "random_forest":
        return RandomForestClassifier(n_jobs=N_JOBS, random_state=SEED, **p)
    if model == "linear_svm":
        return SVC(kernel="linear", max_iter=SVM_MAX_ITER, random_state=SEED, **p)
    if model == "rbf_svm":
        return SVC(kernel="rbf", max_iter=SVM_MAX_ITER, random_state=SEED, **p)
    if model == "mlp":
        p["hidden_layer_sizes"] = tuple(p["hidden_layer_sizes"])
        return MLPClassifier(max_iter=500, random_state=SEED, **p)
    if model == "xgboost":
        p.pop("class_weight", None)   # xu ly bang sample_weight trong fit_model
        return XGBClassifier(tree_method="hist", n_jobs=N_JOBS, random_state=SEED, verbosity=0, **p)
    if model == "lightgbm":
        return LGBMClassifier(n_jobs=N_JOBS, random_state=SEED, verbose=-1, **p)
    raise ValueError(model)


def fit_model(model, params, X, y):
    est = build_model(model, params)
    kw = {}
    if model == "xgboost" and params.get("class_weight") == "balanced":
        kw["sample_weight"] = compute_sample_weight("balanced", y)
    t0 = time.perf_counter()
    est.fit(X, y, **kw)
    return est, time.perf_counter() - t0


DEFAULTS = {
    "knn": {"n_neighbors": 5, "weights": "uniform", "metric": "minkowski"},
    "gaussian_nb": {"var_smoothing": 1e-9},
    "logistic_regression": {"C": 1.0, "class_weight": None},
    "decision_tree": {"max_depth": None, "min_samples_leaf": 1, "criterion": "gini", "class_weight": None},
    "random_forest": {"n_estimators": 500, "max_features": "sqrt", "min_samples_leaf": 1,
                      "class_weight": None, "max_samples": None},
    "linear_svm": {"C": 1.0, "class_weight": None},
    "rbf_svm": {"C": 1.0, "gamma": "scale", "class_weight": None},
    "mlp": {"hidden_layer_sizes": [100], "alpha": 1e-4},
    "xgboost": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3, "subsample": 1.0,
                "colsample_bytree": 1.0, "class_weight": None},
    "lightgbm": {"n_estimators": 100, "num_leaves": 31, "learning_rate": 0.1, "min_child_samples": 20,
                 "class_weight": None},
    # stacking: tham so cua meta-learner XGBoost
    "stacking": {"n_estimators": 100, "max_depth": 6, "learning_rate": 0.3},
}

if QUICK:
    GRIDS = {
        "knn": {"n_neighbors": [5, 11], "weights": ["distance"], "metric": ["minkowski", "cosine"]},
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-3]},
        "logistic_regression": {"C": [0.01, 1.0], "class_weight": ["balanced"]},
        "decision_tree": {"max_depth": [None, 5], "min_samples_leaf": [5], "criterion": ["gini"],
                          "class_weight": ["balanced"]},
        "random_forest": {"n_estimators": [300], "max_features": ["sqrt"], "min_samples_leaf": [1, 5],
                          "class_weight": ["balanced"], "max_samples": [None]},
        "linear_svm": {"C": [1e-3, 1e-1], "class_weight": ["balanced"]},
        "rbf_svm": {"C": [1.0, 10.0], "gamma": ["scale"], "class_weight": ["balanced"]},
        "mlp": {"hidden_layer_sizes": [[100]], "alpha": [1e-4, 1.0]},
        "xgboost": {"n_estimators": [200], "max_depth": [3], "learning_rate": [0.1], "subsample": [0.8],
                    "colsample_bytree": [0.5], "class_weight": [None, "balanced"]},
        "lightgbm": {"n_estimators": [200], "num_leaves": [15], "learning_rate": [0.1],
                     "min_child_samples": [10], "class_weight": [None, "balanced"]},
        "stacking": {"n_estimators": [100], "max_depth": [3], "learning_rate": [0.1]},
    }
else:
    GRIDS = {
        "knn": {"n_neighbors": [3, 5, 7, 11, 15, 21], "weights": ["uniform", "distance"],
                "metric": ["minkowski", "cosine"]},                                    # 24
        "gaussian_nb": {"var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1]},         # 6
        "logistic_regression": {"C": [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0],
                                "class_weight": [None, "balanced"]},                   # 12
        "decision_tree": {"max_depth": [None, 3, 5, 10, 20], "min_samples_leaf": [1, 5, 10],
                          "criterion": ["gini", "entropy"],
                          "class_weight": [None, "balanced"]},                         # 60
        "random_forest": {"n_estimators": [500], "max_features": ["sqrt", "log2", 0.3],
                          "min_samples_leaf": [1, 5, 10], "class_weight": ["balanced"],
                          "max_samples": [None, 0.5]},                                 # 18
        "linear_svm": {"C": [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0],
                       "class_weight": [None, "balanced"]},                            # 12
        "rbf_svm": {"C": [0.1, 1.0, 10.0, 100.0], "gamma": ["scale", 1e-4, 1e-3, 1e-2],
                    "class_weight": [None, "balanced"]},                               # 32
        "mlp": {"hidden_layer_sizes": [[64], [256], [256, 64]],
                "alpha": [1e-4, 1e-2, 1.0, 10.0]},                                     # 12
        "xgboost": {"n_estimators": [200, 500], "max_depth": [3, 6], "learning_rate": [0.05, 0.1],
                    "subsample": [0.8], "colsample_bytree": [0.5, 1.0],
                    "class_weight": [None, "balanced"]},                               # 32
        "lightgbm": {"n_estimators": [200, 500], "num_leaves": [7, 15], "learning_rate": [0.05, 0.1],
                     "min_child_samples": [5, 20], "class_weight": [None, "balanced"]}, # 32
        "stacking": {"n_estimators": [100, 300], "max_depth": [2, 3],
                     "learning_rate": [0.05, 0.1]},                                    # 8
    }


def configs_of(model):
    grid = GRIDS[model]
    cfgs = [dict(zip(grid, v)) for v in itertools.product(*grid.values())]
    return cfgs if DEFAULTS[model] in cfgs else [DEFAULTS[model]] + cfgs


CONFIGS = {m: configs_of(m) for m in MODELS_TO_RUN}
n_cfg = sum(len(c) for c in CONFIGS.values())
print(pd.Series({m: len(c) for m, c in CONFIGS.items()}).to_string())
print(f"Tong: {n_cfg} cau hinh x {N_FOLDS} fold x {len(FS_RUNS)} phuong phap FS "
      f"= {n_cfg * N_FOLDS * len(FS_RUNS)} lan huan luyen (chua tinh CV noi bo cua stacking)")

### Stacking: base MLP, RF, KNN, SVM → meta XGBoost

In [ ]:
STACK_BASES = ["mlp", "random_forest", "knn", "svm"]   # "svm" = linear_svm / rbf_svm tot hon tren val
STACK_CV = 5


def stack_method(model):
    return "decision_function" if model in ("linear_svm", "rbf_svm") else "predict_proba"


def base_output(est, model, X):
    return getattr(est, stack_method(model))(X).reshape(len(X), -1)


class StackedModel:
    # base learner da fit tren toan bo train + meta XGBoost fit tren du doan out-of-fold
    def __init__(self, bases, meta):
        self.bases, self.meta = bases, meta

    def _meta_features(self, X):
        return np.hstack([base_output(est, m, X) for m, est in self.bases])

    def predict(self, X):
        return self.meta.predict(self._meta_features(X))

    def predict_proba(self, X):
        return self.meta.predict_proba(self._meta_features(X))


def fit_stack_bases(bases, X, y):
    # bases: [(model, params)] -> (base da fit, ma tran out-of-fold cho meta, thoi gian)
    n_splits = max(2, min(STACK_CV, np.bincount(y).min()))   # lop hiem (vd LUSC stage IV ~5 mau)
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=SEED)
    t0 = time.perf_counter()
    oof, fitted = [], []
    for model, params in bases:
        oof.append(cross_val_predict(build_model(model, params), X, y, cv=cv,
                                     method=stack_method(model)).reshape(len(y), -1))
        fitted.append((model, build_model(model, params).fit(X, y)))
    return fitted, np.hstack(oof), time.perf_counter() - t0


def fit_stacking(stack_state, meta_params, y):
    fitted, oof, base_s = stack_state
    meta, meta_s = fit_model("xgboost", meta_params, oof, y)
    return StackedModel(fitted, meta), base_s + meta_s

## 5. Độ đo

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, cohen_kappa_score, roc_auc_score,
                             confusion_matrix)

METRIC_NAMES = ["accuracy", "balanced_accuracy", "precision_macro", "precision_weighted",
                "recall_macro", "recall_weighted", "f1_macro", "f1_weighted", "mcc",
                "cohen_kappa", "roc_auc"]


def get_scores(est, X):
    # xac suat (hoac decision_function voi SVC) de tinh ROC-AUC; cot j <-> lop ma hoa j
    if hasattr(est, "predict_proba"):
        return est.predict_proba(X)
    d = est.decision_function(X)
    return np.column_stack([-d, d]) if d.ndim == 1 else d


def roc_auc_ovr_macro(y, scores):
    # one-vs-rest, trung binh macro; bo qua lop khong co mat trong y (AUC khong xac dinh)
    aucs = [roc_auc_score(y == c, scores[:, c]) for c in range(scores.shape[1])
            if 0 < (y == c).sum() < len(y)]
    return float(np.mean(aucs)) if aucs else np.nan


def compute_metrics(y, p, scores):
    kw = dict(zero_division=0)
    return {
        "accuracy": accuracy_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, p),
        "precision_macro": precision_score(y, p, average="macro", **kw),
        "precision_weighted": precision_score(y, p, average="weighted", **kw),
        "recall_macro": recall_score(y, p, average="macro", **kw),
        "recall_weighted": recall_score(y, p, average="weighted", **kw),
        "f1_macro": f1_score(y, p, average="macro", **kw),
        "f1_weighted": f1_score(y, p, average="weighted", **kw),
        "mcc": matthews_corrcoef(y, p),
        "cohen_kappa": cohen_kappa_score(y, p),
        "roc_auc": roc_auc_ovr_macro(y, scores),
    }


# Do do chon cau hinh tren val (chi dung nhan du doan)
SCORERS = {
    "f1_macro": lambda y, p: f1_score(y, p, average="macro", zero_division=0),
    "balanced_accuracy": balanced_accuracy_score,
    "mcc": matthews_corrcoef,
    "accuracy": accuracy_score,
}
select_score = SCORERS[SELECT_METRIC]


def evaluate(est, X, y):
    t0 = time.perf_counter()
    p = est.predict(X)
    infer_s = time.perf_counter() - t0
    return compute_metrics(y, p, get_scores(est, X)), infer_s, p

## 6. Grid search trên 5 fold

Mỗi (phương pháp FS, fold, mô hình): thử mọi cấu hình, fit trên `train`, chọn theo `SELECT_METRIC` trên `val`,
rồi đánh giá cấu hình tốt nhất trên `val` và `test`. `train_time_s` = thời gian fit cấu hình tốt nhất
(stacking: gồm CV out-of-fold + fit base + fit meta), `infer_time_test_s` = thời gian `predict` trên test,
`tuning_time_s` = tổng thời gian dò lưới của mô hình đó. Mỗi mô hình xong được ghi ngay vào `RESULTS_CSV`.

In [ ]:
def grid_search(fit_fn, configs, X_va, y_va):
    best, t0 = None, time.perf_counter()
    for params in configs:
        est, fit_s = fit_fn(params)
        score = select_score(y_va, est.predict(X_va))
        if best is None or score > best["score"]:
            best = {"params": params, "est": est, "fit_s": fit_s, "score": score}
    best["tuning_s"] = time.perf_counter() - t0
    return best


def make_row(stem, info, r, model, best, n_configs, sets, classes, genes):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = sets
    m_va, _, _ = evaluate(best["est"], X_va, y_va)
    m_te, infer_s, p_te = evaluate(best["est"], X_te, y_te)
    cm = confusion_matrix(y_te, p_te, labels=np.arange(len(classes)))
    return {
        "n_features": N_FEATURES_TAG, "fs_group": FS_GROUP, "fs_method": info["method"], "fs_run": stem,
        "dataset": info["dataset"], "fold": r, "model": model, "n_genes": len(genes),
        "n_train": len(y_tr), "n_val": len(y_va), "n_test": len(y_te),
        "best_params": json.dumps(best["params"]), "n_configs": n_configs,
        "train_time_s": best["fit_s"], "tuning_time_s": best["tuning_s"],
        "infer_time_test_s": infer_s, "infer_time_per_sample_ms": 1e3 * infer_s / len(y_te),
        **{f"val_{k}": v for k, v in m_va.items()},
        **{f"test_{k}": v for k, v in m_te.items()},
        "class_labels": json.dumps([STAGE_NAMES.get(c, str(c)) for c in classes]),
        "confusion_matrix_test": json.dumps(cm.tolist()),
    }


def append_row(row):
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a", index=False, header=not os.path.exists(RESULTS_CSV))

In [ ]:
prev = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame(
    columns=["fs_run", "fold", "model", "best_params", f"val_{SELECT_METRIC}"])
done = set(zip(prev["fs_run"], prev["fold"].astype(int), prev["model"]))
if done:
    print(f"Tiep tuc: da co {len(done)} dong trong {RESULTS_CSV}")

t_start = time.time()
out_of_time = False
todo_all = [(k, r, m) for k in FS_RUNS for r in range(1, N_FOLDS + 1) for m in MODELS_TO_RUN
            if (k, r, m) not in done]
print(f"Con lai {len(todo_all)} (fs_run, fold, model) can chay")
for stem, info in sorted(FS_RUNS.items(), key=lambda kv: (kv[1]["dataset"], kv[0])):
    if out_of_time:
        break
    for r in range(1, N_FOLDS + 1):
        if out_of_time:
            break
        todo = [m for m in MODELS_TO_RUN if (stem, r, m) not in done]
        if not todo:
            continue
        genes = read_genes(info["folds"][r])
        fold = load_dataset(info["dataset"])[r]
        (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = [get_xy(fold[s], genes) for s in ["train", "val", "test"]]

        # Ma hoa nhan ve 0..K-1 theo cac lop cua train (XGBoost yeu cau; GSE68465 khong co stage IV)
        classes = np.unique(y_tr)
        assert set(y_va) <= set(classes) and set(y_te) <= set(classes), f"{stem} fold {r}: val/test co lop la"
        y_tr, y_va, y_te = [np.searchsorted(classes, y) for y in (y_tr, y_va, y_te)]
        sets = [(X_tr, y_tr), (X_va, y_va), (X_te, y_te)]

        # cau hinh tot nhat cua fold nay (tu lan chay truoc neu co) -> base learner cho stacking
        fold_best = {row.model: (json.loads(row.best_params), getattr(row, f"val_{SELECT_METRIC}"))
                     for row in prev[(prev.fs_run == stem) & (prev.fold == r)].itertuples()}

        for model in [m for m in todo if m != "stacking"] + [m for m in todo if m == "stacking"]:
            if time.time() - NOTEBOOK_T0 > TIME_BUDGET_H * 3600:
                out_of_time = True
                print(f"!! Het TIME_BUDGET_H={TIME_BUDGET_H}h -- dung truoc {stem} fold {r} {model}. "
                      f"Dua zip ket qua vao PREV_RESULTS va chay lai de tiep tuc.")
                break
            t0 = time.time()
            if model == "stacking":
                svm = max(["linear_svm", "rbf_svm"], key=lambda m: fold_best.get(m, (None, -1))[1])
                bases = []
                for b in STACK_BASES:
                    b = svm if b == "svm" else b
                    if b not in fold_best:
                        print(f"  [stacking] chua co cau hinh tot nhat cua {b}, dung DEFAULTS")
                    bases.append((b, fold_best.get(b, (DEFAULTS[b], None))[0]))
                state = fit_stack_bases(bases, X_tr, y_tr)
                best = grid_search(lambda p: fit_stacking(state, p, y_tr), CONFIGS[model], X_va, y_va)
                best["params"] = {"meta_xgboost": best["params"], "bases": {b: p for b, p in bases}}
            else:
                best = grid_search(lambda p: fit_model(model, p, X_tr, y_tr), CONFIGS[model], X_va, y_va)
            row = make_row(stem, info, r, model, best, len(CONFIGS[model]), sets, classes, genes)
            append_row(row)
            fold_best[model] = (best["params"], row[f"val_{SELECT_METRIC}"])
            done.add((stem, r, model))
            print(f"{stem} | fold {r} | {model:20s} val F1 {row['val_f1_macro']:.3f} | "
                  f"test F1 {row['test_f1_macro']:.3f} acc {row['test_accuracy']:.3f} "
                  f"AUC {row['test_roc_auc']:.3f} | {time.time() - t0:.0f}s "
                  f"(tong {(time.time() - t_start) / 60:.1f} phut)", flush=True)

_res = pd.read_csv(RESULTS_CSV)
_have = set(zip(_res["fs_run"], _res["fold"].astype(int), _res["model"]))
_missing = [(k, r, m) for k in FS_RUNS for r in range(1, N_FOLDS + 1) for m in MODELS_TO_RUN
            if (k, r, m) not in _have]
print("Xong grid search:", RESULTS_CSV, f"| {len(_res)} dong")
print("CON THIEU:", _missing if _missing else "khong -- da du")

## 7. Tổng hợp: trung bình qua 5 fold

`summary_<nhom>.csv`: mỗi dòng = (phương pháp FS, bộ, mô hình); cột độ đo = trung bình 5 fold,
cột `*_std` = độ lệch chuẩn của độ đo test, `confusion_matrix_test_sum` = cộng confusion matrix test
của 5 fold (mỗi mẫu làm test đúng 1 lần).

In [ ]:
res = pd.read_csv(RESULTS_CSV)
KEYS = ["n_features", "fs_group", "fs_method", "dataset", "model"]
TIME_COLS = ["train_time_s", "tuning_time_s", "infer_time_test_s", "infer_time_per_sample_ms"]
VAL_COLS = [f"val_{m}" for m in METRIC_NAMES]
TEST_COLS = [f"test_{m}" for m in METRIC_NAMES]


def sum_cm(s):
    mats = [np.array(json.loads(x)) for x in s]
    return json.dumps(np.sum(mats, axis=0).tolist()) if len({m.shape for m in mats}) == 1 else ""


g = res.groupby(KEYS, sort=False)
summary = g[TEST_COLS].mean()
for c in TEST_COLS:
    summary[f"{c}_std"] = g[c].std()
summary = summary[[c for t in TEST_COLS for c in (t, f"{t}_std")]]
summary = summary.join(g[VAL_COLS + TIME_COLS].mean())
summary["n_folds"] = g.size()
summary["class_labels"] = g["class_labels"].first()
summary["confusion_matrix_test_sum"] = g["confusion_matrix_test"].apply(sum_cm)
summary = summary.reset_index()
summary.to_csv(SUMMARY_CSV, index=False)
print("Da luu:", SUMMARY_CSV, summary.shape)

# Bang nhanh: test macro-F1 trung binh, hang = (phuong phap FS, bo), cot = mo hinh
display(summary.pivot_table(index=["fs_method", "dataset"], columns="model", values="test_f1_macro")
        .reindex(columns=[m for m in MODELS_TO_RUN if m in set(summary.model)]).round(3))

In [ ]:
import shutil
zip_out = shutil.make_archive(os.path.join("/kaggle/working", f"classification_{N_FEATURES_TAG}_{FS_GROUP}"),
                              "zip", OUT_DIR)
print("Da nen:", zip_out, f"({os.path.getsize(zip_out) / 1e6:.2f} MB)")

# Kiem tra: moi fs_run phai du N_FOLDS x so mo hinh dong
_chk = pd.read_csv(RESULTS_CSV).groupby("fs_run").size()
print(_chk.to_string())
print(f"Tong {_chk.sum()} dong, {len(_chk)} fs_run (day du = {len(_chk) * N_FOLDS * len(MODELS_TO_RUN)})")
